In [1]:
import numpy as np
import pandas as pd

from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GroupKFold, HalvingGridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

import os

### Get the File Location of the Dataset

In [2]:
dataset_path = os.getcwd()
# print(dataset_path)
dataset_path = os.path.join(dataset_path, "Final_Dataset")
# print(os.listdir(dataset_path))
dataset_path = os.path.join(dataset_path, os.listdir(dataset_path)[0])
print(dataset_path)

data = pd.read_csv(dataset_path)
print(data)
X = data.iloc[:, 0:data.shape[1]-1].values
# print(X)
y = data.iloc[:, data.shape[1]-1].values

c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\Final_Dataset\concatenated_data.csv
       PatientID  time_sin  time_cos     glucose  calories  heart_rate  steps  \
0              1 -0.984808  0.173648  332.000000   6.35950   82.322835     34   
1              1 -0.980785  0.195090  326.000000   7.72800   83.740157      0   
2              1 -0.976296  0.216440  330.000000   4.74950   80.525180      0   
3              1 -0.971342  0.237686  324.000000   6.35950   89.129032     20   
4              1 -0.965926  0.258819  306.000000   5.15200   92.495652      0   
...          ...       ...       ...         ...       ...         ...    ...   
89673         25 -0.382683 -0.923880  124.000000   4.94670   76.708333      0   
89674         25 -0.402747 -0.915311  121.666667   5.84610   72.475000      0   
89675         25 -0.422618 -0.906308  119.333333   4.58694   69.587302      0   
89676         25 -0.442289 -0.896873  117.000000   4.67688   75.775000     

### Create the SVR Model

#### Resources:
1. https://www.analyticsvidhya.com/blog/2020/03/support-vector-regression-tutorial-for-machine-learning/
2. https://scikit-learn.org/1.5/modules/generated/sklearn.svm.SVR.html
3. Picking which kernel should be used for linear and non-linear relationships: https://www.geeksforgeeks.org/support-vector-regression-svr-using-linear-and-non-linear-kernels-in-scikit-learn/
4. https://www.geeksforgeeks.org/time-series-forecasting-with-support-vector-regression/
5. https://www.geeksforgeeks.org/ml-feature-scaling-part-2/

Note: Our dataset may need to use a kernel that is applicable for non-linear relationships since time-series data is prone to alot of changes. 

### Creating the Train-Test Split

#### Resources:
1. https://www.geeksforgeeks.org/how-to-generate-a-train-test-split-based-on-a-group-id/#importance-of-groupbased-splitting

### Generating the Metrics:

#### Resources:
1. https://scikit-learn.org/1.5/modules/model_evaluation.html#r2-score-the-coefficient-of-determination
2. https://scikit-learn.org/1.5/modules/model_evaluation.html#mean-squared-error

In [ ]:
##Without Using MinMaxScalar

##Default Hyperparmeters: C = 1
svr_model = SVR(kernel = 'rbf')
##Overall there are 25 groups since there are 25 patient IDs
kfold_split = GroupKFold(n_splits = 5)
groups = X[:, 0]
r2_scores, rmse_vals = [], []
# Apply splits while ensuring patient grouping is maintained
split_num = 1
for train_index, test_index in kfold_split.split(data, groups = groups):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    # print(f"Unique classes in TRAIN: {np.unique(X_train[:,0])}")
    # print(f"Unique classes in TEST: {np.unique(X_test[:,0])}")
    # print(f"TRAIN shapes: X: {X_train.shape}, y: {y_train.shape}")
    # print(f"TEST shapes: X: {X_test.shape}, y: {y_test.shape}")

    X_train, X_test = X_train[:, 1:], X_test[:, 1:]

    #Fit the model
    svr_model.fit(X_train, y_train)

    # Predict
    y_pred = svr_model.predict(X_test)

    # Calculate r2
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Split Num: {split_num}")
    print(f"R2 Score: {r2}")
    print(f"RMSE: {np.sqrt(mse)}")
    r2_scores.append(r2)
    rmse_vals.append(np.sqrt(mse))
    split_num += 1

Unique classes in TRAIN: [ 2.  3.  4.  5.  6.  9. 10. 11. 12. 13. 15. 16. 18. 19. 20. 21. 22. 23.
 24. 25.]
Unique classes in TEST: [ 1.  7.  8. 14. 17.]
TRAIN shapes: X: (71452, 11), y: (71452,)
TEST shapes: X: (18226, 11), y: (18226,)
Split Num: 1
R2 Score: 0.3695008705920302
RMSE: 5.507925371296757
Unique classes in TRAIN: [ 1.  2.  3.  5.  6.  7.  8. 10. 12. 13. 14. 15. 16. 17. 18. 19. 21. 22.
 23. 25.]
Unique classes in TEST: [ 4.  9. 11. 20. 24.]
TRAIN shapes: X: (71737, 11), y: (71737,)
TEST shapes: X: (17941, 11), y: (17941,)


In [ ]:
##Using MinMaxScalar for Standardization
svr_model = SVR(kernel = 'rbf')
scaler = MinMaxScaler()

##Overall there are 25 groups since there are 25 patient IDs
ts_split = GroupKFold(n_splits = 5)
groups = X[:, 0]
min_max_r2_scores, min_max_rmse_vals = [], []
# Apply splits while ensuring patient grouping is maintained
split_num = 1
for train_index, test_index in ts_split.split(data, groups = groups):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    print(f"Unique classes in TRAIN: {np.unique(X_train[:,0])}")
    print(f"Unique classes in TEST: {np.unique(X_test[:,0])}")
    print(f"TRAIN shapes: X: {X_train.shape}, y: {y_train.shape}")
    print(f"TEST shapes: X: {X_test.shape}, y: {y_test.shape}")

    X_train, X_test = X_train[:, 1:], X_test[:, 1:]

    scaled_X_train = scaler.fit_transform(X_train)
    scaled_X_test = scaler.transform(X_test)

    #Fit the model
    svr_model.fit(scaled_X_train, y_train)

    # Predict
    y_pred = svr_model.predict(scaled_X_test)

    # Calculate r2
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Split Num: {split_num}")
    print(f"R2 Score: {r2}")
    print(f"RMSE: {np.sqrt(mse)}")
    min_max_r2_scores.append(r2)
    min_max_rmse_vals.append(np.sqrt(mse))
    split_num += 1

In [ ]:
##Using MinMaxScalar and HyperParameter Optimization


### Generating the Plots:

#### Resources:
1. https://www.analyticsvidhya.com/blog/2020/03/support-vector-regression-tutorial-for-machine-learning/